# La órbita de gauge bajo cuantización: llanura, cola y la ley del producto

Cuaderno de lectura de la medición `gauge-cuantizacion/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [cuarenta-y-cuatro-de-cinco-mil](https://manpla.net/posts/cuarenta-y-cuatro-de-cinco-mil/). Carga los ficheros de al lado —o los descarga del repositorio—, muestra la ficha de procedencia y dibuja dos figuras con matplotlib a secas. Solo lee; `generar.py` vuelve a agregar las tablas desde el conjunto publicado en Hugging Face.

El dato crudo no está aquí: son 170 016 medidas del barrido principal y 642 048 del de condicionamiento controlado, y viven en [ManPla/gauge-orbit-quantization-results](https://huggingface.co/datasets/ManPla/gauge-orbit-quantization-results), junto al depósito [10.5281/zenodo.22904208](https://doi.org/10.5281/zenodo.22904208).

*Reading notebook for this measurement: loads the files next to it (or from the repository), prints the provenance note and draws two plain matplotlib figures. It only reads; the raw sweeps live in the linked Hugging Face dataset.*

In [ ]:
import json
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/gauge-cuantizacion/"


def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    return pd.read_csv(p, **kw) if p.exists() else pd.read_csv(RAW + nombre, **kw)


def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


RES = json.loads(texto("resumen.json"))
print(RES["fuente"], "·", RES["deposito"], "· extraído", RES["extraido"])

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## La llanura ortogonal

Por celda (capa, cabeza, anchura), el cociente entre el peor error ortogonal muestreado y el de los pesos sin tocar. El listón preregistrado llama «estrecha» a la celda que queda bajo ×1,25: lo cumplen todas.

In [ ]:
orb = leer("orbita-ortogonal.csv")
print(orb[orb.bits == 4][["modelo", "pregunta", "celdas", "estrechas", "p50", "min", "max"]]
      .to_string(index=False))
orb

## La cola de los gauges generales

Fuera del subgrupo ortogonal, el error crece con la fuerza del cambio de coordenadas. La mediana sube de forma ordenada; el máximo se dispara.

In [ ]:
cola = leer("cola-gl.csv")
c4 = cola[cola.bits == 4]
fig, ax = plt.subplots(figsize=(9, 4.5))
for m, mk in (("pythia", "o"), ("vitb", "s")):
    d = c4[c4.modelo == m].sort_values("escala", ascending=False)
    ax.plot(d.escala.astype(str), d.mediana, marker=mk, label=f"{m} · mediana")
    ax.plot(d.escala.astype(str), d.maximo, marker=mk, ls=":", label=f"{m} · máximo")
ax.set_yscale("log")
ax.set_xlabel("escala del gauge (32 suave → 2 fuerte)")
ax.set_ylabel("error relativo del circuito")
ax.legend(); plt.tight_layout(); plt.show()
c4

## La ley del producto

El factor `p` de la cota —el producto de las normas de los dos factores transformados, normalizado por el de la identidad— contra el cociente de errores. La cota predice exponente uno sin ajustar nada; el número de condición, la medida clásica, no lo consigue.

In [ ]:
bins = leer("cota-producto.csv")
b4 = bins[bins.bits == 4]
fig, ax = plt.subplots(figsize=(9, 5))
lo, hi = b4.p_mediano.min() * 0.7, b4.p_mediano.max() * 1.5
ax.plot([lo, hi], [lo, hi], ls="--", color="grey", label="pendiente uno")
ax.scatter(b4.p_mediano, b4.ratio_mediano, s=70, label="contra el producto de normas")
ax.scatter(b4.kappa_mediano, b4.ratio_mediano, s=60, marker="s",
           label="contra el número de condición")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("factor de la cota · número de condición")
ax.set_ylabel("error con el cambio ÷ error sin él")
ax.legend(); plt.tight_layout(); plt.show()
print("ajuste publicado — producto:", RES["cota"]["exponente_producto"],
      "· condición:", RES["cota"]["exponente_kappa"])

## El contraste extremo a extremo

Las siete comparaciones pareadas sobre las mismas 5 000 imágenes: exactitud de cada condición, diferencia en puntos, discordancias, McNemar y acuerdo de clase predicha. La última fila es la que carga el peso del artículo.

In [ ]:
e2e = leer("contraste-e2e.csv")
e2e

In [ ]:
a = RES["e2e"]
print("sin cuantizar, el cambio de coordenadas acuerda en el",
      f'{a["acuerdo_fp32"] * 100:.0f} % de las imágenes',
      f'({a["discordantes_fp32"]} discordancias)')
print("a int4, acuerda en", a["imagenes_de_acuerdo_int4"], "de",
      RES["imagenes_validacion"], "imágenes · caída de", a["caida_pp"], "puntos")